# 00 — Environment and Reproducibility

Purpose: Record software, hardware, seeds, source checksums, and the exact 20-model registry.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


## Fixed protocol

Seeds: 17, 42, 2026. Primary criterion: macro per-battery MAE; tie-breakers: RMSE, serialized size, then CPU latency.

In [2]:
import json, os, platform
import numpy as np, pandas as pd, sklearn, scipy, psutil, torch, tensorflow as tf
import xgboost, lightgbm
from src.models.catalog import model_catalog_records

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor(),
    "logical_cpu_count": os.cpu_count(),
    "ram_gib": round(psutil.virtual_memory().total / 1024**3, 2),
    "cuda_available": torch.cuda.is_available(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pytorch": torch.__version__,
    "tensorflow": tf.__version__,
    "xgboost": xgboost.__version__,
    "lightgbm": lightgbm.__version__,
    "seeds": [17, 42, 2026],
    "primary_metric": "macro_per_battery_mae",
}
(RESULTS / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
catalog = pd.DataFrame(model_catalog_records())
catalog.to_csv(RESULTS / "model_catalog.csv", index=False)
display(catalog)
environment

,model_id,display_name,family,input_kind,framework
0,extra_trees,Extra Trees,classical,tabular,scikit-learn
1,gradient_boosting,Gradient Boosting,classical,tabular,scikit-learn
2,random_forest,Random Forest,classical,tabular,scikit-learn
3,xgboost,XGBoost,classical,tabular,xgboost
4,lightgbm,LightGBM,classical,tabular,lightgbm
5,svr,Support Vector Regression,classical,tabular,scikit-learn
6,ridge,Ridge Regression,classical,tabular,scikit-learn
7,knn,K-Nearest Neighbours,classical,tabular,scikit-learn
8,vanilla_lstm,Vanilla LSTM,recurrent,sequence,pytorch
9,bidirectional_lstm,Bidirectional LSTM,recurrent,sequence,pytorch


{'python': '3.12.10',
 'platform': 'Windows-11-10.0.26200-SP0',
 'processor': 'Intel64 Family 6 Model 183 Stepping 1, GenuineIntel',
 'logical_cpu_count': 20,
 'ram_gib': 15.71,
 'cuda_available': False,
 'numpy': '2.1.3',
 'pandas': '2.3.3',
 'scikit_learn': '1.9.1',
 'scipy': '1.18.1',
 'pytorch': '2.14.0+cpu',
 'tensorflow': '2.20.0',
 'xgboost': '2.1.4',
 'lightgbm': '4.7.0',
 'seeds': [17, 42, 2026],
 'primary_metric': 'macro_per_battery_mae'}

In [3]:
checksum_file = PROJECT_ROOT / "datasets" / "checksums.sha256"
print(checksum_file.read_text(encoding="utf-8") if checksum_file.exists() else "Run the benchmark downloader first.")

84d88cb98a43c02049f16c78806a78ec19084776ea5df8a4b7c2f8f7ba4589e5  calce/CS2_35.zip
2e674f58bb6dc005c73bef10e87edbbd97afd041dde77a1d461a3c4ecac1b027  calce/CS2_36.zip
8ed8aecbcbea2bcb17f455923a974be01ea21d7389210e1912d177de5b6af4a5  calce/CS2_37.zip
98787427c10478425efd4f54d823805724daff1d2fcbd822b3a2d7006a1051d8  calce/CS2_38.zip
82302a7db4fc1b34e0b6676326610438d43b816bdf11a69d1d012a464ef2f92e  nasa/5_Battery_Data_Set.zip
a8f0b928f4ea77d7ba98b97194b78cc77336f2338fee8b57573d043f0cf26781  oxford/Oxford_Battery_Degradation_Dataset_1.mat

